In [1]:
import torch
import os
import numpy as np
import pickle as pkl

In [2]:
from torch.utils.data import StackDataset

DATASET_PATHS = os.path.join('..','data')

TRAIN_DATASET_PATH = os.path.join(DATASET_PATHS, 'train.pickle')

with open(TRAIN_DATASET_PATH, "rb") as f:
    train_dataset = pkl.load(f)

train_data = torch.tensor(
    np.array(list(train_dataset["sensor_data"]), dtype=np.float32),
)
train_data = train_data.moveaxis(-1,-2)

train_labels = torch.tensor(
    np.array(list(train_dataset["label"])),
)

train_dataset = StackDataset(train_data, train_labels)

/tmp/ipykernel_510/2858958482.py:8: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  train_dataset = pkl.load(f)


In [3]:
if torch.cuda.is_available():
    print("CUDA is available!")
    DEVICE = torch.device("cuda")
else:
    print("CUDA is not available!")
    DEVICE = torch.device("cpu")

CUDA is available!


## Define the random-split benchmark function

By default we take a 80-20 split from the `train.pickle` and train the model 20 times with a constant epoch.

In [4]:
from torch.utils.data import random_split, DataLoader
from utils import run

def random_split_benchmark(Classifier, Optimizer, LossFunction, epochs, dataset, device, model_args = {}, optimizer_args = {}, batch_size=32, log=False, runs=20, p=0.8, generator=None):
    losses = []
    accs = []

    for i in range(runs):
        print(f"Starting run {i+1}...")
        [train_dataset, test_dataset] = random_split(dataset, [p, 1.0-p], generator=generator)
        loss, acc = run(Classifier, Optimizer, LossFunction, epochs, DataLoader(train_dataset, batch_size=batch_size) , DataLoader(test_dataset, batch_size=batch_size), device, model_args = {}, optimizer_args = {}, log=log)
        print(f"Run {i+1}: loss {loss} acc {acc}")
        losses.append(loss)
        accs.append(acc)

    return np.array(losses), np.array(accs)


### Choose the model and parameters

As pointed out by the report, many combinations of parameters were tried. All models perform reasonably well on the provided `train.pickle`, even if random-splitting was introduced. 
The main problem lies in detecting overfitting or rather, good performance on `test.pickle`.

In my opinion though, `CNNClassifier` performed the best on both datasets. Choose some reasonable parameters and try it yourself!

In [5]:
import torch.nn as nn
import torch.optim as optim
from task1.cnn.model import CNNClassifier
from task1.fcn.model import FullyConnectedModel
from task1.lstm.model import LSTM
from utils import run

runs = 10

Classifier = CNNClassifier
Optimizer = optim.Adam
LossFunction = nn.CrossEntropyLoss

optimizer_args = { 'lr' : 0.001 }
epochs = 7

In [6]:
random_split_benchmark(
    Classifier,
    Optimizer,
    LossFunction,
    epochs,
    train_dataset,
    DEVICE,
    optimizer_args=optimizer_args,
    runs=runs,
    p=0.8
)

Starting run 1...


/root/.venv/lib/python3.12/site-packages/torch/nn/modules/linear.py:125: UserWarning: Attempting to use hipBLASLt on an unsupported architecture! Overriding blas backend to hipblas (Triggered internally at ../aten/src/ATen/Context.cpp:296.)
  return F.linear(input, self.weight, self.bias)


Run 1: loss 0.03731640476537983 acc 0.991358024691358
Starting run 2...
Run 2: loss 0.0515290583472377 acc 0.9962962962962963
Starting run 3...
Run 3: loss 0.007595554618509831 acc 1.0
Starting run 4...
Run 4: loss 0.18125554601120028 acc 0.9938271604938271
Starting run 5...
Run 5: loss 0.0793775337232984 acc 0.9950617283950617
Starting run 6...
Run 6: loss 0.040490521856779114 acc 0.9962962962962963
Starting run 7...
Run 7: loss 0.15419908352196215 acc 0.9481481481481482
Starting run 8...
Run 8: loss 0.030582871327153694 acc 0.9975308641975309
Starting run 9...
Run 9: loss 0.061950730741093 acc 0.9888888888888889
Starting run 10...
Run 10: loss 0.10949266896637755 acc 0.9975308641975309


(array([0.0373164 , 0.05152906, 0.00759555, 0.18125555, 0.07937753,
        0.04049052, 0.15419908, 0.03058287, 0.06195073, 0.10949267]),
 array([0.99135802, 0.9962963 , 1.        , 0.99382716, 0.99506173,
        0.9962963 , 0.94814815, 0.99753086, 0.98888889, 0.99753086]))

### Detect Overfitting by using `test.pickle`

In [7]:
TEST_DATASET_PATH = os.path.join(DATASET_PATHS,'test.pickle')

with open(TEST_DATASET_PATH, "rb") as f:
    test_dataset = pkl.load(f)

test_data = torch.tensor(
    np.array(list(test_dataset["sensor_data"]), dtype=np.float32),
    device=DEVICE
)
test_data = test_data.moveaxis(-1,-2)

test_labels = torch.tensor(
    np.array(list(test_dataset["label"])),
    device=DEVICE
)

test_dataset = StackDataset(test_data, test_labels)
test_dataloader = DataLoader(test_dataset, batch_size=32)

/tmp/ipykernel_510/3373523511.py:4: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  test_dataset = pkl.load(f)


In [8]:
from utils import benchmark

benchmark(
    Classifier,
    Optimizer,
    LossFunction,
    epochs,
    DataLoader(train_dataset, batch_size=32),
    DataLoader(test_dataset, batch_size=32),
    DEVICE,
    optimizer_args=optimizer_args,
    runs=runs,
)

Starting run 1...
Run 1: loss 2.2310963498724328 acc 0.5924075924075924
Starting run 2...
Run 2: loss 2.155078299514778 acc 0.5034965034965035
Starting run 3...
Run 3: loss 3.0390349628922944 acc 0.29270729270729273
Starting run 4...
Run 4: loss 2.227665918452161 acc 0.4745254745254745
Starting run 5...
Run 5: loss 2.083744142915343 acc 0.5924075924075924
Starting run 6...
Run 6: loss 1.7585148925666922 acc 0.5924075924075924
Starting run 7...
Run 7: loss 1.8559749187170327 acc 0.5924075924075924
Starting run 8...
Run 8: loss 1.8388400306473007 acc 0.5934065934065934
Starting run 9...
Run 9: loss 1.9729734168543325 acc 0.5924075924075924
Starting run 10...
Run 10: loss 1.796605067772346 acc 0.5594405594405595


(array([2.23109635, 2.1550783 , 3.03903496, 2.22766592, 2.08374414,
        1.75851489, 1.85597492, 1.83884003, 1.97297342, 1.79660507]),
 array([0.59240759, 0.5034965 , 0.29270729, 0.47452547, 0.59240759,
        0.59240759, 0.59240759, 0.59340659, 0.59240759, 0.55944056]))